# **Tugas Akhir**   
### Analisis Perbandingan Model Deep Learning untuk Klasifikasi Emosi pada Teks Opini Publik Berbahasa Indonesia

2372023 - Eben Praus Yosia



---



# Part 2: Preprocessing "Minimal"

Cleaning minimal teks, lalu penanganan data kosong, konflik label, dan duplikat.
- Input : `data/raw_combined.csv` (hasil Part 1)
- Output: `data/clean.csv` (kolom `text`, `label`) -> nantinya akan dipakai Part 3 (split data)

In [1]:
import re
import pandas as pd

IN_PATH  = "data/raw_combined.csv"
OUT_PATH = "data/clean.csv"

TEXT_COL, LABEL_COL = "text", "label"
pd.set_option("display.max_colwidth", 130)

df_raw = pd.read_csv(IN_PATH)
print("Data mentah:", df_raw.shape)
df_raw.head(3)

Data mentah: (7105, 2)


,text,label
0,pagi2 udah di buat emosi :),anger
1,"kok stabilitas negara, memange 10 thn negara tdk aman, bahkan sby menyuburkan ormas2 radikal, intoleran, teroris, yg berafilia...",anger
2,dah lah emosi mulu liat emyu,anger


### Step 1 - Aturan cleaning
Yang dilakukan: case folding, hapus URL, hapus mention (@user), hapus karakter non-print (karakter kontrol)

Yang sengaja tidak dilakukan: hapus tanda baca, hapus pengulangan huruf, hapus emoji/emotikon, normalisasi kata slang

In [2]:
URL_RE     = re.compile(r"(?:https?://\S+|www\.\S+)", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
# Karakter kontrol (C0 & C1) kecuali tab/newline
CTRL_RE    = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]")
SPACE_RE   = re.compile(r"\s+")

def clean_text(text: str) -> str:
    """Preprocessing MINIMAL untuk klasifikasi emosi.

    SENGAJA TIDAK dilakukan (karena bisa membawa informasi emosi):
      - Menghapus tanda baca  : "!!!" (marah/senang), "???" (bingung/takut), "..." (sedih/ragu)
      - Menghapus pengulangan huruf : "bangeeet", "sediiih", "aaaa" (penegasan intensitas emosi)
      - Menghapus emoji/emotikon    : ":(" ":)" "<3" langsung menandai emosi
      - Normalisasi slang/singkatan : "bgt", "wkwk", "anjir", "gpp" adalah ekspresi khas emosi
      - Karakter format (mis. zero-width joiner) TIDAK dihapus agar emoji gabungan tidak rusak
    Catatan: TF-IDF hanya dipakai di baseline (Tahap 4). Preprocessing ini sama untuk semua model.
    """
    t = str(text).lower()            # 1. case folding
    t = URL_RE.sub(" ", t)           # 2. hapus URL
    t = MENTION_RE.sub(" ", t)       # 3. hapus mention
    t = CTRL_RE.sub(" ", t)          # 4. hapus karakter non-print
    t = SPACE_RE.sub(" ", t).strip() # "perapian": spasi ganda/tab/newline -> 1 spasi
    return t

# Uji cepat pada kalimat buatan sendiri
contoh = "Sedih BANGEEET!!! :( @budi cek https://t.co/abc  \x9d  wkwk"
print("sebelum:", repr(contoh))
print("sesudah:", repr(clean_text(contoh)))

sebelum: 'Sedih BANGEEET!!! :( @budi cek https://t.co/abc  \x9d  wkwk'
sesudah: 'sedih bangeeet!!! :( cek wkwk'


### Step 2 - Terapkan & lihat sebelum vs sesudah dicleaning

In [3]:
df = df_raw.copy()
df["text_raw"] = df[TEXT_COL].astype(str)
df[TEXT_COL]   = df["text_raw"].apply(clean_text)

# Berapa teks yang terkena tiap aturan? (dihitung pada raw text)
raw = df["text_raw"]
n_url   = raw.apply(lambda s: URL_RE.search(s) is not None).sum()
n_ment  = raw.str.contains(MENTION_RE.pattern, regex=True).sum()
n_ctrl  = raw.str.contains(CTRL_RE.pattern, regex=True).sum()
n_upper = (raw != raw.str.lower()).sum()
n_berubah = (df["text_raw"] != df[TEXT_COL]).sum()

pd.DataFrame({
    "aturan": ["Mengandung huruf kapital (case folding)", "Mengandung URL", "Mengandung mention (@)",
               "Mengandung karakter non-print", "TOTAL teks yang berubah (semua aturan)"],
    "jumlah teks": [n_upper, n_url, n_ment, n_ctrl, n_berubah],
})

,aturan,jumlah teks
0,Mengandung huruf kapital (case folding),22
1,Mengandung URL,0
2,Mengandung mention (@),0
3,Mengandung karakter non-print,81
4,TOTAL teks yang berubah (semua aturan),112


In [4]:
# Contoh sebelum/sesudah untuk tiap aturan (maks. 3 per aturan)
def tampil(nama, mask, n=3):
    sub = df[mask].head(n)
    print(f"\n=== {nama} ({mask.sum()} teks) ===")
    for _, r in sub.iterrows():
        print("  SEBELUM:", repr(r["text_raw"][:150]))
        print("  SESUDAH:", repr(r[TEXT_COL][:150]))

tampil("URL",               raw.apply(lambda s: URL_RE.search(s) is not None))
tampil("Mention",           raw.str.contains(MENTION_RE.pattern, regex=True))
tampil("Karakter non-print", raw.str.contains(CTRL_RE.pattern, regex=True))
tampil("Huruf kapital",     raw != raw.str.lower())


=== URL (0 teks) ===

=== Mention (0 teks) ===

=== Karakter non-print (81 teks) ===
  SEBELUM: 'gabole marah \x9d nanti cepett tua'
  SESUDAH: 'gabole marah nanti cepett tua'
  SEBELUM: 'diorang ni semua dah lama nak borak dengan kau. tak fun kata diorang, kalau tak dengar kau buat lawak bodoh satu hari. \x9d tu dulu la jepp. haha. \x9d kito'
  SESUDAH: 'diorang ni semua dah lama nak borak dengan kau. tak fun kata diorang, kalau tak dengar kau buat lawak bodoh satu hari. tu dulu la jepp. haha. kitorang'
  SEBELUM: 'yhh marah \x9d mulu lo, cepet tua mampus'
  SESUDAH: 'yhh marah mulu lo, cepet tua mampus'

=== Huruf kapital (22 teks) ===
  SEBELUM: 'jangan sampai rakyatÂ\xa0benci'
  SESUDAH: 'jangan sampai rakyatâ benci'
  SEBELUM: 'rakyat sudah ribut dari kemarin, lalu dpr baru kaget sekarang. mereka digaji dengan uang rakyat kemarin ngapain aja? bukannya dpr harusnya check balan'
  SESUDAH: 'rakyat sudah ribut dari kemarin, lalu dpr baru kaget sekarang. mereka digaji dengan uang rak

In [5]:
# Bukti bahwa tanda baca, pengulangan huruf, emotikon, slang tidak dihapus:
# dgn cara ambil contoh yang mengandung pola tsb, tampilkan hasil akhirnya
pola = {
    "pengulangan huruf (xxx)": r"(\w)\1{2,}",
    "tanda baca berulang":     r"[!?.]{2,}",
    "emotikon :( :) <3":       r":\(|:\)|<3|:\'\(",
    "slang (bgt/wkwk/gpp)":    r"\b(bgt|wkwk\w*|gpp|anjir)\b",
}
for nama, p in pola.items():
    m = df[TEXT_COL].apply(lambda s: re.search(p, s) is not None)
    print(f"\n--- {nama}: {m.sum()} teks masih mengandungnya setelah cleaning ---")
    for t in df.loc[m, TEXT_COL].head(2): print("  ", t[:130])


--- pengulangan huruf (xxx): 151 teks masih mengandungnya setelah cleaning ---
   ahahaaa khas buzzerp jokowi nuduh org mau, dituduh marah
   khai lu bego bgt tolol brengski oon gblok aaaak ksel bgt gw kesell

--- tanda baca berulang: 1017 teks masih mengandungnya setelah cleaning ---
   kok stabilitas negara, memange 10 thn negara tdk aman, bahkan sby menyuburkan ormas2 radikal, intoleran, teroris, yg berafiliasi k
   1-10!! marah aku

--- emotikon :( :) <3: 206 teks masih mengandungnya setelah cleaning ---
   pagi2 udah di buat emosi :)
   paling benci kalo lagi diem terus panik sendiri gatau bingung kenapa :)

--- slang (bgt/wkwk/gpp): 739 teks masih mengandungnya setelah cleaning ---
   mas ngelonte dulu mas, biar otaknya jadi seger lg, bagi org kafir seperti lu ngewe dgn lonte ibadah yg utama. punya tolol kok keba
   mau marah bangeett kenapa orang orang jahat bgt si


### Step 3 - Hapus data kosong, konflik label, dan duplikat
Hal ini dilakukan sesudah cleaning, karena cleaning bisa memunculkan duplikat/kosong baru.  
Urutan:
1. Hapus teks kosong (mis. tweet yang isinya hanya URL/mention).
2. Hapus semua baris dari teks yang punya label berbeda (ambigu, tidak bisa dipercaya kelasnya).
3. Hapus duplikat (teks + label sama), sisakan satu.

In [6]:
log = [("Awal (setelah cleaning)", len(df))]

# 1. Data kosong
df = df[df[TEXT_COL].str.len() > 0]
log.append(("Setelah hapus teks kosong", len(df)))

# 2. Konflik label: teks sama tapi label berbeda -> buang semua barisnya
n_label_per_teks = df.groupby(TEXT_COL)[LABEL_COL].transform("nunique")
konflik = df[n_label_per_teks > 1]
print(f"Teks konflik: {konflik[TEXT_COL].nunique()} teks unik ({len(konflik)} baris). Contoh:")
display(konflik.sort_values(TEXT_COL)[[TEXT_COL, LABEL_COL]].head(8))
df = df[n_label_per_teks == 1]
log.append(("Setelah hapus teks konflik label", len(df)))

# 3. Duplikat (teks + label sama)
df = df.drop_duplicates(subset=[TEXT_COL, LABEL_COL], keep="first")
log.append(("Setelah hapus duplikat", len(df)))

df = df.reset_index(drop=True)
alur = pd.DataFrame(log, columns=["tahap", "jumlah baris"])
alur["dibuang"] = (-alur["jumlah baris"].diff()).fillna(0).astype(int)
alur

Teks konflik: 11 teks unik (23 baris). Contoh:


,text,label
6608,enak dong,neutral
2447,enak dong,joy
5900,"kabar gembira untuk kita semua, kulit manggis kini ada ekstraknya, mastin hadir dan rawat tubuh kita jadikan hari ini hari mas...",neutral
2388,"kabar gembira untuk kita semua, kulit manggis kini ada ekstraknya, mastin hadir dan rawat tubuh kita jadikan hari ini hari mas...",joy
6401,kaget kan lo,neutral
1518,kaget kan lo,fear
232,kegaduhan yg ditimbulkan dan efek demo bersilit silitnya sampe skrg ga ilang2. meski pd akhirnya mrk marah krn kecewa jebul si...,anger
6832,kegaduhan yg ditimbulkan dan efek demo bersilit silitnya sampe skrg ga ilang2. meski pd akhirnya mrk marah krn kecewa jebul si...,neutral


,tahap,jumlah baris,dibuang
0,Awal (setelah cleaning),7105,0
1,Setelah hapus teks kosong,7105,0
2,Setelah hapus teks konflik label,7082,23
3,Setelah hapus duplikat,7031,51


### Step 4 - Distribusi kelas sebelum vs sesudah

In [7]:
sebelum = df_raw[LABEL_COL].value_counts().rename("sebelum")
sesudah = df[LABEL_COL].value_counts().rename("sesudah")
cmp = pd.concat([sebelum, sesudah], axis=1)
cmp["dibuang"] = cmp["sebelum"] - cmp["sesudah"]
cmp["persen_sesudah (%)"] = (cmp["sesudah"] / cmp["sesudah"].sum() * 100).round(2)
cmp.loc["TOTAL"] = [cmp["sebelum"].sum(), cmp["sesudah"].sum(), cmp["dibuang"].sum(), 100.0]
cmp

,sebelum,sesudah,dibuang,persen_sesudah (%)
label,,,,
neutral,2002.0,1986.0,16.0,28.25
joy,1275.0,1255.0,20.0,17.85
anger,1135.0,1123.0,12.0,15.97
sad,1022.0,1008.0,14.0,14.34
fear,911.0,905.0,6.0,12.87
love,760.0,754.0,6.0,10.72
TOTAL,7105.0,7031.0,74.0,100.00


### Step 5 - Simpan hasil

In [8]:
df[[TEXT_COL, LABEL_COL]].to_csv(OUT_PATH, index=False)
print("Tersimpan:", OUT_PATH, "|", df[[TEXT_COL, LABEL_COL]].shape)

# Pengecekan akhir
assert df[TEXT_COL].isna().sum() == 0 and (df[TEXT_COL].str.len() > 0).all()
assert not df.duplicated(subset=[TEXT_COL]).any() # tidak ada teks duplikat sama sekali
assert df[LABEL_COL].nunique() == 6
print("Semua pengecekan lolos: tidak ada teks kosong/duplikat, 6 kelas utuh.")

Tersimpan: data/clean.csv | (7031, 2)
Semua pengecekan lolos: tidak ada teks kosong/duplikat, 6 kelas utuh.


### Notes Part 2:
- Jumlah data akhir: 7.031 teks
- Teks yang berubah karena cleaning: 112
- Jumlah dibuang (kosong | konflik | duplikat): 0 | 23 | 51
- Kelas terbanyak | tersedikit setelah cleaning: neutral (1.986) | love (754)